# Fase 3: Data Preparation (Avanzada / Datos Reales)
En esta fase descargamos los datos **macroeconómicos de la Reserva Federal (FRED) a partir de 2009** y armamos la Matriz con TODO el Análisis HEAVY del periodo 2009-2026 (crisis del narco, shocks financieros y COVID). Luego haremos el merge con ICT.

In [ ]:
import pandas as pd
import numpy as np
import os
import datetime

print('Librerías importadas exitosamente.')

### 1. Ingest: Descarga de Variables Globales (Reserva Federal - FRED)

In [ ]:
# ¡ACTUALIZADO A 2009 PARA COINCIDIR CON LA DATA MÁS ANTIGUA QUE TENEMOS (LLEGADAS MIGRATORIAS)!
start_date = '2009-01-01'

series = {
    'UNRATE': 'tasa_desempleo_usa',                 
    'USINFO': 'empleo_sector_tech_usa',             
    'UMCSENT': 'sentimiento_consumidor_michi',      
    'CPIAUCSL': 'inflacion_usa_cpi',                
    'DCOILWTICO': 'precio_petroleo_wti',            
    'DSPIC96': 'ingreso_disponible_usa'             
}

print('Descargando datos macro desde FRED mediante acceso directo a CSV a partir de 2009...')
df_list = []

for s_id, s_name in series.items():
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={s_id}'
    df_temp = pd.read_csv(url, parse_dates=['observation_date'], na_values='.')
    df_temp = df_temp.rename(columns={'observation_date': 'DATE'})
    
    df_temp = df_temp[df_temp['DATE'] >= start_date].copy()
    df_temp[s_id] = pd.to_numeric(df_temp[s_id], errors='coerce')
    
    df_temp.set_index('DATE', inplace=True)
    df_temp = df_temp.resample('ME').mean()  
    df_temp.rename(columns={s_id: s_name}, inplace=True)
    
    df_list.append(df_temp)

df_macro = pd.concat(df_list, axis=1)
df_macro.reset_index(inplace=True)
df_macro['DATE'] = df_macro['DATE'].dt.strftime('%Y-%m-%d')

print(f'Descarga completa. Total de meses descargados: {df_macro.shape[0]}')
display(df_macro.head())

### 2. Feature Engineering: INTEGRACIÓN DEL ANÁLISIS 2009-2026
Se incorporan TODOS los hallazgos del último reporte: Epidemia H1N1 (2009), Crisis Seguridad/Homicidios (2017-2018), Pandemia, y Guerra Narco (2023-2026).

In [ ]:
### A. MACRO-EVENTOS GLOBALES / MERCADOS EMISORES

# 1. Pánico Sanitario H1N1 (Abril 2009 a Diciembre 2009)
df_macro['brote_h1n1_2009'] = df_macro['DATE'].apply(lambda x: 1 if '2009-04-01' <= x <= '2009-12-31' else 0)

# 2. COVID-19 Agudo (Cierres totales Marzo 2020 - Dic 2021)
df_macro['covid_agudo_cr_mundial'] = df_macro['DATE'].apply(lambda x: 1 if '2020-03-01' <= x <= '2021-12-31' else 0)

# 3. Guerra Ucrania y Crisis Energética Europea (2022)
df_macro['guerra_ucrania_activa'] = df_macro['DATE'].apply(lambda x: 1 if x >= '2022-02-01' else 0)

# 4. Inflación Energética Global Post-Pandemia (2021 - 2023)
df_macro['crisis_precios_energia_global'] = df_macro['DATE'].apply(lambda x: 1 if '2021-01-01' <= x <= '2023-12-31' else 0)

# 5. Recesión y Caída GDP per Cápita Canadá (A partir de mid-2022)
df_macro['stress_economico_canada'] = df_macro['DATE'].apply(lambda x: 1 if x >= '2022-06-01' else 0)

# 6. USA Election Seasons (Abarcando 2012, 2016, 2020, 2024)
def usa_elections(fecha):
    if ('2012-11' <= fecha <= '2013-01' or 
        '2016-11' <= fecha <= '2017-01' or 
        '2020-11' <= fecha <= '2021-01' or 
        '2024-11' <= fecha <= '2025-01'):
        return 1
    return 0
df_macro['usa_election_season'] = df_macro['DATE'].apply(usa_elections)


### B. SHOCKS REGIONALES CLIMÁTICOS (ESTADOS TOP USA)

# 7. Incendios California (Ago-Nov en 2018, 2020, 2021, 2024)
def incendios_california(fecha):
    if (('2018-08' <= fecha <= '2018-11') or ('2020-08' <= fecha <= '2020-11') or 
        ('2021-08' <= fecha <= '2021-11') or ('2024-08' <= fecha <= '2024-11')):
        return 1
    return 0
df_macro['shock_california_fires'] = df_macro['DATE'].apply(incendios_california)

# 8. Marea Roja en Florida (Cancelaciones 2018)
df_macro['shock_florida_redtide'] = df_macro['DATE'].apply(lambda x: 1 if '2018-07-01' <= x <= '2018-12-31' else 0)

# 9. Huracanes Mayores Florida (Ian Sep-Nov 2022, Milton Oct-Dic 2024)
df_macro['shock_florida_hurricanes'] = df_macro['DATE'].apply(lambda x: 1 if ('2022-09' <= x <= '2022-11') or ('2024-10' <= x <= '2024-12') else 0)

### C. MICRO-EVENTOS COSTA RICA / GUANACASTE (INCLUYENDO CRISIS NARCO)

# 10. Huelga Nacional Reforma Fiscal CR (Afectó vías en CR)
df_macro['huelga_nacional_cr_2018'] = df_macro['DATE'].apply(lambda x: 1 if '2018-09-01' <= x <= '2018-11-30' else 0)

# 11. COLAPSO AEROPUERTO LIR Y LLUVIAS (Doble impacto en Guanacaste: Noviembre y Diciembre 2024)
df_macro['colapso_pista_lir_2024'] = df_macro['DATE'].apply(lambda x: 1 if x == '2024-11-30' or x == '2024-11-01' else 0)

# 12. Epidemia de Zika en Latam (2016)
df_macro['brote_zika_latam'] = df_macro['DATE'].apply(lambda x: 1 if '2016-02-01' <= x <= '2016-11-30' else 0)

# 13. OLA INSEGURIDAD 1 (Homicidios record >12/100k y asesinatos turistas)
df_macro['crisis_seguridad_2017_2018'] = df_macro['DATE'].apply(lambda x: 1 if '2017-01-01' <= x <= '2018-12-31' else 0)

# 14. OLA INSEGURIDAD 2 / GUERRA NARCO (Enero 2023 hasta la actualidad 2026)
df_macro['crisis_narcotrafico_2023_2026'] = df_macro['DATE'].apply(lambda x: 1 if x >= '2023-01-01' else 0)

print('✅ MATRIZ 2009-2026 AÑADIDA: Total de 14 perturbaciones (Shocks) cuantificadas para el modelo.')
display(df_macro.tail(10))

### 3. Integración de la DATA y Generación del Pipeline Cascade
Aquí leeremos los excels de ICT ('Ocupación 2018-2024' y 'Llegadas 2009-2026'). Utilizaremos las Llegadas para enseñar a predecir la Ocupación.

In [ ]:
print("A la espera de los archivos de ICT en data/raw/ para fusionarlos y validar el Merge")